In [113]:
%pip install --upgrade setuptools
%restart_python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 20.8 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 74.0.0
    Not uninstalling setuptools at /usr/local/lib/python3.12/dist-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-ca10460b-4d62-4f55-9cce-2ce610d7711d
    Can't uninstall 'setuptools'. No files were found to uninstall.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pmdarima 2.1.1 requires statsmodels>=0.14.5, but you have statsmodels 0.14.4 which is incompatible.
spacy 3.8.14 requires confection<2.0.0,>=1.3.2, but you have confection 0.1.5 which is incompatible.
spacy 3.8.14 requires srsly<3.0.0,>=2.5.3, but you have srsly 2.5.1 which is incompatible.
spacy 3.8.14 requires weasel<2.0.0,>=1.0.0, but you have weasel 0.4.1 which is incompatible.
thinc 8.3.13 requires blis<1.

Error: The "path" argument must be of type string. Received type undefined

## Loading Databricks Tables (╯'□')╯︵ ┻━┻

In [ ]:
import pandas as pd
import pyspark.pandas as ps
import os
import glob
import shutil

spark.sql("USE stg_ccc")

In [ ]:
spark.sql("""
    SELECT case_id 
    FROM ds_ccc.kg_sulo_sample_cases_table
""").createOrReplaceTempView("sampled_case_ids")

spark.sql("""
    SELECT DISTINCT c.case_id
    FROM sampled_case_ids c
    JOIN stg_ccc.sf_survey_subject ss ON c.case_id = ss.subjectid
    JOIN stg_ccc.sf_survey_invitation si ON ss.parentid = si.id
    JOIN stg_ccc.sf_survey_question_response sqr ON si.id = sqr.invitationid
    WHERE sqr.questionid = '0Ku7S000000RnrrSAC'
    AND sqr.numbervalue IS NOT NULL
""").createOrReplaceTempView("cases_with_surveys")

case_filtered = spark.sql("""
    SELECT case_id, person_account_id, business_account_id, case_vehicle_structure_id, case_closure_type, case_type, case_service_type, support_level, customer_country, market_country, case_reason, case_priority, case_description, case_solution_final_statement, case_handling_time_sec, case_closed_date, case_created_date, case_last_modified_date, case_conversation_amt, case_session_amt, is_closed_flg 
    FROM stg_ccc.w_case
    WHERE case_id IN (SELECT case_id FROM cases_with_surveys)
    LIMIT 100000
""")

case_filtered.createOrReplaceTempView("sampled_cases")
case_df = case_filtered.pandas_api()

In [116]:
survey_case = spark.sql("""
    SELECT 
        c.case_id,
        sqr.id AS survey_response_id,
        sqr.questionid,
        CASE 
            WHEN CAST(sqr.numbervalue AS FLOAT) >= 3 THEN 1 
            ELSE 0 
        END AS numbervalue,
        si.id AS invitation_id
    FROM sampled_cases c
    JOIN stg_ccc.sf_survey_subject ss ON c.case_id = ss.subjectid
    JOIN stg_ccc.sf_survey_invitation si ON ss.parentid = si.id
    JOIN stg_ccc.sf_survey_question_response sqr ON si.id = sqr.invitationid
    WHERE sqr.questionid = '0Ku7S000000RnrrSAC'
""")

survey_df = survey_case.pandas_api()

In [117]:
account_filtered = spark.sql("""
    SELECT a.id, a.salutation, a.type, a.createddate, a.lastmodifieddate, 
           a.core_country__c, a.core_preferredlanguage__c 
    FROM stg_ccc.sf_account a
    WHERE a.isdeleted = 'N'
    AND (
        a.id IN (SELECT person_account_id FROM sampled_cases WHERE person_account_id IS NOT NULL)
        OR 
        a.id IN (SELECT business_account_id FROM sampled_cases WHERE business_account_id IS NOT NULL)
    )
""")

account_df = account_filtered.pandas_api()

In [118]:
task_filtered = spark.sql("""
    WITH select_cases AS (
        SELECT case_id
        FROM sampled_cases
    )
    SELECT
        t.whatid AS case_id,
        t.completeddatetime AS event_date,
        t.id AS task_id,
        t.type AS task_type,
        t.subject AS task_subject,
        t.description AS task_description,
        t.ishighpriority,
        t.calldurationinseconds
    FROM
        westeurope_cac_dwh_prd.sf_ccc.task t
    WHERE 
        t.whatid IN (SELECT case_id FROM select_cases)
""")

task_df = task_filtered.pandas_api()

In [119]:
history_filtered = spark.sql("""
    WITH select_cases AS (
        SELECT case_id
        FROM sampled_cases
    )
    SELECT
        ch.id as modification_id,
        ch.caseid AS case_id,
        ch.createddate AS event_date,
        ch.field AS changed_field,
        ch.oldvalue AS old_value,
        ch.newvalue AS new_value
    FROM 
        westeurope_cac_dwh_prd.sf_ccc.casehistory ch
    WHERE 
        ch.caseid IN (SELECT case_id FROM select_cases)
""")

history_df = history_filtered.pandas_api()

In [120]:
vehicle_filtered = spark.sql("SELECT vehicle_structure_id, vehicle_structure_name, brand_name, class_name, serie_name, product_group_name FROM stg_ccc.w_vehicle_structure")

vehicle_df = vehicle_filtered.pandas_api()

## Turtle Construction 🐢

In [0]:
%pip install owlready2<=0.47

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.3/27.3 MB 24.7 MB/s eta 0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for owlready2: filename=owlready2-0.47-cp312-cp312-linux_x86_64.whl size=24584810 sha256=ef81f97363d0143e542d19e775dc0bbb1d220930a78cd18caf8e273f42d26312
  Stored in directory: /root/.cache/pip/wheels/a8/81/8a/f76909ab0a8a17ce3735a6b9501344c545be5098d54b66adfc
Successfully built owlready2
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import rdflib
import numpy as np
import re
import numpy as np
import urllib.parse
import os
import glob
import shutil

from owlready2 import *
from rdflib.plugins.sparql import prepareQuery
from rdflib import Graph
from graphviz import Digraph

In [ ]:
def format_uri(base_namespace: str, entity_type: str, identifier: ps.Series) -> ps.Series:
    """
    Vectorized URI generation utilizing distributed pandas-on-Spark Series.
    Aggressively sanitizes whitespace and illegal URI characters.
    """
    clean_id = identifier.astype(str).str.strip().str.replace(r'[\s<>"{}|\\^`]+', '_', regex=True)
    return "<" + base_namespace + entity_type + "/" + clean_id + ">"

def format_literal(values: ps.Series, datatype: str = "http://www.w3.org/2001/XMLSchema#string") -> ps.Series:
    """
    Vectorized generation of typed literals utilizing distributed pandas-on-Spark Series.
    Safely escapes double quotes and newlines to prevent N-Triples parser failures.
    """
    safe_vals = values.astype(str).str.replace('\\', '\\\\') \
                                  .str.replace('"', '\\"') \
                                  .str.replace('\n', '\\n') \
                                  .str.replace('\r', '')
    return "\"" + safe_vals + "\"^^<" + datatype + ">"

In [ ]:
def populate_baseline_kg_explicit(df: ps.DataFrame, entity_column: str, entity_type: str, mappings: dict) -> list:
    base_namespace = "https://w3id.org/sulo/schema/resource/ontology-schema/29dbbab2-7fa8-458c-a7d6-a973aebced40/"
    rdf_type = "<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>"
    
    clean_entities = df[entity_column].astype(str).str.strip().str.replace(r'[\s<>"{}|\\^`]+', '_', regex=True)
    subjects = "<" + base_namespace + entity_type + "/" + clean_entities + ">"
    
    series_list = []
    
    entity_class_uri = "<" + base_namespace + entity_type + ">"
    type_triples = subjects + " " + rdf_type + " " + entity_class_uri + " ."
    series_list.append(type_triples.drop_duplicates())
    count = 1
    for column, (predicate_uri, is_object_property, target_type) in mappings.items():
        if column not in df.columns:
            continue
            
        valid_mask = df[column].notna()
        valid_subjects = subjects[valid_mask]
        valid_values = df.loc[valid_mask, column].astype(str)
        
        predicates = "<" + predicate_uri + ">"
        
        if is_object_property:
            valid_uri_values = valid_values.str.strip().str.replace(r'[\s<>"{}|\\^`]+', '_', regex=True)
            objects = "<" + base_namespace + target_type + "/" + valid_uri_values + ">"
            
            relation_triples = valid_subjects + " " + predicates + " " + objects + " ."
            series_list.append(relation_triples)
            
            target_class_uri = "<" + base_namespace + target_type + ">"
            target_type_triples = objects + " " + rdf_type + " " + target_class_uri + " ."
            series_list.append(target_type_triples.drop_duplicates())
            
        else:
            safe_literals = valid_values.str.replace('\\', '\\\\').str.replace('"', '\\"').str.replace('\n', '\\n').str.replace('\r', '')
            objects = "\"" + safe_literals + "\"^^<" + target_type + ">"
            
            relation_triples = valid_subjects + " " + predicates + " " + objects + " ."
            series_list.append(relation_triples)
            
    return series_list

In [ ]:
BASE_URI = "https://w3id.org/sulo/schema/resource/ontology-schema/29dbbab2-7fa8-458c-a7d6-a973aebced40/"
# Format: "DataFrame_Column": ("Predicate_URI", is_object_property, "Target_Class_or_Datatype")

# Executes on 'survey_case' DataFrame with entity_column="case_id" and entity_type="Case"
survey_mappings = {
    "numbervalue": (f"{BASE_URI}hasSurveyRating", True, "SurveyRating")
}

# Executes on 'account_filtered' DataFrame with entity_column="id" and entity_type="Account"
account_mappings = {
    "salutation": (f"{BASE_URI}hasSalutation", False, "http://www.w3.org/2001/XMLSchema#string"),
    "type": (f"{BASE_URI}hasAccountType", True, "AccountType"),
    "createddate": (f"{BASE_URI}hasCreateDate", False, "http://www.w3.org/2001/XMLSchema#dateTime"),
    "lastmodifieddate": (f"{BASE_URI}hasAccountLastModifiedDate", False, "http://www.w3.org/2001/XMLSchema#dateTime"),
    "core_country__c": (f"{BASE_URI}hasCountry", True, "Country"),
    "core_preferredlanguage__c": (f"{BASE_URI}hasLanguage", True, "Language")
}

# To populate the Task attributes, execute on 'task_filtered' with entity_column="task_id" and entity_type="Task"
task_mappings = {
    "event_date": (f"{BASE_URI}hasTaskCompletedDate", False, "http://www.w3.org/2001/XMLSchema#dateTime"),
    "task_type": (f"{BASE_URI}hasTaskType", True, "TaskType"),
    "task_subject": (f"{BASE_URI}hasTaskSubject", False, "http://www.w3.org/2001/XMLSchema#string"),
    "task_description": (f"{BASE_URI}hasTaskDescription", False, "http://www.w3.org/2001/XMLSchema#string"),
    "ishighpriority": (f"{BASE_URI}hasPriorityLevel", True, "PriorityLevel"),
    "calldurationinseconds": (f"{BASE_URI}hasTaskDurationSec", False, "http://www.w3.org/2001/XMLSchema#integer")
}

# To link the Case to the Task, execute a secondary pass on 'task_filtered' with entity_column="case_id" and entity_type="Case"
case_to_task_mappings = {
    "task_id": (f"{BASE_URI}hasTask", True, "Task")
}

history_mappings = {
    "event_date": (f"{BASE_URI}hasModificationDate", False, "http://www.w3.org/2001/XMLSchema#dateTime"),
    "changed_field": (f"{BASE_URI}hasModifiedField", False, "http://www.w3.org/2001/XMLSchema#string"),
    "old_value": (f"{BASE_URI}hasPreviousValue", False, "http://www.w3.org/2001/XMLSchema#string"),
    "new_value": (f"{BASE_URI}hasNewValue", False, "http://www.w3.org/2001/XMLSchema#string")
}

# To link the Case to the Modification, execute a secondary pass on 'history_filtered' with entity_column="case_id" and entity_type="Case"
case_to_history_mappings = {
    "modification_id": (f"{BASE_URI}hasCaseModification", True, "CaseModification")
}

# Executes on 'vehicle_filtered' DataFrame with entity_column="vehicle_structure_id" and entity_type="Vehicle"
vehicle_mappings = {
    "vehicle_structure_name": (f"{BASE_URI}hasVehicleStructureName", False, "http://www.w3.org/2001/XMLSchema#string"),
    "brand_name": (f"{BASE_URI}hasBrand", True, "Brand"),
    "class_name": (f"{BASE_URI}hasVehicleClass", True, "VehicleClass"),
    "serie_name": (f"{BASE_URI}hasVehicleSeries", True, "VehicleSeries"),
    "product_group_name": (f"{BASE_URI}hasProductGroup", True, "ProductGroup")
}

case_mappings = {
    "person_account_id": (f"{BASE_URI}hasAccount", True, "Account"),
    "business_account_id": (f"{BASE_URI}hasAccount", True, "Account"),
    "case_vehicle_structure_id": (f"{BASE_URI}hasVehicle", True, "Vehicle"),
    "case_priority": (f"{BASE_URI}hasPriorityLevel", True, "PriorityLevel"),
    "case_closure_type": (f"{BASE_URI}hasCaseClosureType", True, "CaseClosureType"),
    "case_type": (f"{BASE_URI}hasCaseType", True, "CaseType"),
    "case_service_type": (f"{BASE_URI}hasServiceType", True, "ServiceType"),
    "support_level": (f"{BASE_URI}hasSupportLevel", True, "SupportLevel"),
    "case_reason": (f"{BASE_URI}hasCaseReason", True, "CaseReason"),
    "is_closed_flg": (f"{BASE_URI}hasCaseStatus", True, "CaseStatus"),

    # Datatype Properties (column_name: (predicate_uri, is_object_property, datatype_uri))
    "case_description": (f"{BASE_URI}hasCaseDescription", False, "http://www.w3.org/2001/XMLSchema#string"),
    "case_solution_final_statement": (f"{BASE_URI}hasCaseSolutionFinalStatement", False, "http://www.w3.org/2001/XMLSchema#string"),
    "case_handling_time_sec": (f"{BASE_URI}hasCaseHandlingTimeSec", False, "http://www.w3.org/2001/XMLSchema#integer"),
    "case_closed_date": (f"{BASE_URI}hasCaseClosedDate", False, "http://www.w3.org/2001/XMLSchema#dateTime"),
    "case_created_date": (f"{BASE_URI}hasCreateDate", False, "http://www.w3.org/2001/XMLSchema#dateTime"),
    "case_conversation_amt": (f"{BASE_URI}hasCaseConversationAmount", False, "http://www.w3.org/2001/XMLSchema#integer"),
    "case_session_amt": (f"{BASE_URI}hasCaseSessionAmount", False, "http://www.w3.org/2001/XMLSchema#integer")
}

## Populate Baseline

In [ ]:
baseline_collection = []

baseline_collection.extend(populate_baseline_kg_explicit(survey_df, "case_id", "Case", survey_mappings))
baseline_collection.extend(populate_baseline_kg_explicit(account_df, "id", "Account", account_mappings))
baseline_collection.extend(populate_baseline_kg_explicit(task_df, "task_id", "Task", task_mappings))
baseline_collection.extend(populate_baseline_kg_explicit(task_df, "case_id", "Case", case_to_task_mappings))
baseline_collection.extend(populate_baseline_kg_explicit(history_df, "modification_id", "CaseModification", history_mappings))
baseline_collection.extend(populate_baseline_kg_explicit(history_df, "case_id", "Case", case_to_history_mappings))
baseline_collection.extend(populate_baseline_kg_explicit(vehicle_df, "vehicle_structure_id", "Vehicle", vehicle_mappings))
baseline_collection.extend(populate_baseline_kg_explicit(case_df, "case_id", "Case", case_mappings))

master_baseline_triples_ps = ps.concat(baseline_collection, ignore_index=True)
master_spark_df = master_baseline_triples_ps.to_frame(name="ntriples").to_spark()

volume_base_path = "/Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km"
temp_dir = f"{volume_base_path}/temp_graph_dir"
final_file = f"{volume_base_path}/baseline_populated_graph.nt"

print("Coalescing and writing data (this may take a moment)...")
master_spark_df.coalesce(1).write.mode("overwrite").text(temp_dir)
part_file_path = glob.glob(f"{temp_dir}/part-*")[0]

shutil.move(part_file_path, final_file)

shutil.rmtree(temp_dir)

print(f"Success! Single file generated at: {final_file}")

total_triples = master_spark_df.count()
print(f"Baseline Knowledge Graph successfully populated. Total Triples Generated: {total_triples}")

In [ ]:
def populate_sulo_kg_explicit(df: ps.DataFrame, entity_column: str, entity_type: str, mappings: dict) -> list:
    """
    Transforms tabular data into SULO-compliant N-Triples using vectorized operations.
    Returns a standard Python list containing distributed pandas-on-Spark Series.
    """
    base_namespace = "https://w3id.org/sulo/schema/resource/ontology-schema/29dbbab2-7fa8-458c-a7d6-a973aebced40/"
    sulo_ns = "https://w3id.org/sulo/"
    rdf_type = "<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>"
    
    clean_entities = df[entity_column].astype(str).str.strip().str.replace(r'[\s<>"{}|\\^`]+', '_', regex=True)
    
    subjects = "<" + base_namespace + entity_type + "/" + clean_entities + ">"
    
    series_list = []

    entity_class_uri = "<" + base_namespace + entity_type + ">"
    type_triples = subjects + " " + rdf_type + " " + entity_class_uri + " ."
    
    # Drop duplicates so we only assert the entity's type once, even if the dataframe has multiple rows for it
    series_list.append(type_triples.drop_duplicates())
    
    for column, mapping in mappings.items():
        if column not in df.columns:
            continue
            
        valid_mask = df[column].notna()
        valid_subjects = subjects[valid_mask]
        
        valid_entity_ids = clean_entities[valid_mask]
        
        valid_values = df.loc[valid_mask, column].astype(str)
        
        valid_uri_values = valid_values.str.strip().str.replace(r'[\s<>"{}|\\^`]+', '_', regex=True)
        
        # Escape double quotes and newlines so they don't break N-Triples syntax
        safe_literals = valid_values.str.replace('\\', '\\\\') \
                                    .str.replace('"', '\\"') \
                                    .str.replace('\n', '\\n') \
                                    .str.replace('\r', '')
        
        pattern = mapping["pattern"]
        
        if pattern == "ROLE_CHAIN":
            role_class = mapping["role_class"]
            target_class = mapping["target_class"]
            
            roles = "<" + base_namespace + role_class + "/" + valid_entity_ids + "_" + valid_uri_values + ">"
            targets = "<" + base_namespace + target_class + "/" + valid_uri_values + ">"
            
            target_class_uri = "<" + base_namespace + target_class + ">"
            target_type_triples = targets + " " + rdf_type + " " + target_class_uri + " ."
            series_list.append(target_type_triples.drop_duplicates())
            
            # 1. Subject -> sulo:hasParticipant -> Role
            t1 = valid_subjects + f" <{sulo_ns}hasParticipant> " + roles + " ."
            # 2. Role -> rdf:type -> RoleClass
            t2 = roles + f" {rdf_type} <{base_namespace}{role_class}> ."
            # 3. Role -> sulo:isFeatureOf -> Target Object
            t3 = roles + f" <{sulo_ns}isFeatureOf> " + targets + " ."
            
            series_list.extend([t1, t2, t3])
            
        elif pattern == "SOLID_LITERAL":
            sulo_predicate = mapping["sulo_predicate"]
            intermediate_class = mapping["intermediate_class"]
            datatype = mapping["datatype"]
            
            intermediate_nodes = "<" + base_namespace + intermediate_class + "/" + valid_entity_ids + "_" + column + ">"
            
            literals = "\"" + safe_literals + "\"^^<" + datatype + ">"
            
            # 1. Subject -> sulo_predicate -> Intermediate Node
            t1 = valid_subjects + f" <{sulo_ns}{sulo_predicate}> " + intermediate_nodes + " ."
            # 2. Intermediate Node -> rdf:type -> IntermediateClass
            t2 = intermediate_nodes + f" {rdf_type} <{base_namespace}{intermediate_class}> ."
            # 3. Intermediate Node -> sulo:hasValue -> Typed Literal
            t3 = intermediate_nodes + f" <{sulo_ns}hasValue> " + literals + " ."
            
            series_list.extend([t1, t2, t3])
            
        elif pattern == "DIRECT_OBJECT":
            sulo_predicate = mapping["sulo_predicate"]
            target_class = mapping["target_class"]
            
            targets = "<" + base_namespace + target_class + "/" + valid_uri_values + ">"
            
            target_class_uri = "<" + base_namespace + target_class + ">"
            target_type_triples = targets + " " + rdf_type + " " + target_class_uri + " ."
            series_list.append(target_type_triples.drop_duplicates())
            
            # 1. Subject -> sulo_predicate -> Target Object
            t1 = valid_subjects + f" <{sulo_ns}{sulo_predicate}> " + targets + " ."
            
            series_list.append(t1)
            
    return series_list

In [ ]:
case_sulo_mappings = {
    "person_account_id": {"pattern": "ROLE_CHAIN", "role_class": "AccountRole", "target_class": "Account"},
    "business_account_id": {"pattern": "ROLE_CHAIN", "role_class": "AccountRole", "target_class": "Account"},
    "case_vehicle_structure_id": {"pattern": "ROLE_CHAIN", "role_class": "VehicleRole", "target_class": "Vehicle"},
    
    "case_priority": {"pattern": "DIRECT_OBJECT", "sulo_predicate": "hasFeature", "target_class": "PriorityLevel"},
    "case_closure_type": {"pattern": "DIRECT_OBJECT", "sulo_predicate": "hasFeature", "target_class": "CaseClosureType"},
    "case_type": {"pattern": "DIRECT_OBJECT", "sulo_predicate": "hasFeature", "target_class": "CaseType"},
    "case_service_type": {"pattern": "DIRECT_OBJECT", "sulo_predicate": "hasFeature", "target_class": "ServiceType"},
    "support_level": {"pattern": "DIRECT_OBJECT", "sulo_predicate": "hasFeature", "target_class": "SupportLevel"},
    "case_reason": {"pattern": "DIRECT_OBJECT", "sulo_predicate": "isReferredToIn", "target_class": "CaseReason"},
    "is_closed_flg": {"pattern": "DIRECT_OBJECT", "sulo_predicate": "hasFeature", "target_class": "CaseStatus"},

    "case_description": {"pattern": "SOLID_LITERAL", "sulo_predicate": "isReferredToIn", "intermediate_class": "InformationObject", "datatype": "http://www.w3.org/2001/XMLSchema#string"},
    "case_solution_final_statement": {"pattern": "SOLID_LITERAL", "sulo_predicate": "isReferredToIn", "intermediate_class": "InformationObject", "datatype": "http://www.w3.org/2001/XMLSchema#string"},
    "case_handling_time_sec": {"pattern": "SOLID_LITERAL", "sulo_predicate": "atTime", "intermediate_class": "DurationSec", "datatype": "http://www.w3.org/2001/XMLSchema#integer"},
    "case_closed_date": {"pattern": "SOLID_LITERAL", "sulo_predicate": "atTime", "intermediate_class": "TimeInstant", "datatype": "http://www.w3.org/2001/XMLSchema#dateTime"},
    "case_created_date": {"pattern": "SOLID_LITERAL", "sulo_predicate": "atTime", "intermediate_class": "TimeInstant", "datatype": "http://www.w3.org/2001/XMLSchema#dateTime"},
    "case_last_modified_date": {"pattern": "SOLID_LITERAL", "sulo_predicate": "atTime", "intermediate_class": "TimeInstant", "datatype": "http://www.w3.org/2001/XMLSchema#dateTime"},
    "case_conversation_amt": {"pattern": "SOLID_LITERAL", "sulo_predicate": "hasFeature", "intermediate_class": "Amount", "datatype": "http://www.w3.org/2001/XMLSchema#integer"},
    "case_session_amt": {"pattern": "SOLID_LITERAL", "sulo_predicate": "hasFeature", "intermediate_class": "Amount", "datatype": "http://www.w3.org/2001/XMLSchema#integer"}
}

account_sulo_mappings = {
    "salutation": {"pattern": "SOLID_LITERAL", "sulo_predicate": "hasFeature", "intermediate_class": "InformationObject", "datatype": "http://www.w3.org/2001/XMLSchema#string"},
    "type": {"pattern": "DIRECT_OBJECT", "sulo_predicate": "hasFeature", "target_class": "AccountType"},
    "createddate": {"pattern": "SOLID_LITERAL", "sulo_predicate": "atTime", "intermediate_class": "TimeInstant", "datatype": "http://www.w3.org/2001/XMLSchema#dateTime"},
    "lastmodifieddate": {"pattern": "SOLID_LITERAL", "sulo_predicate": "atTime", "intermediate_class": "TimeInstant", "datatype": "http://www.w3.org/2001/XMLSchema#dateTime"},
    "core_country__c": {"pattern": "DIRECT_OBJECT", "sulo_predicate": "hasFeature", "target_class": "Country"},
    "core_preferredlanguage__c": {"pattern": "DIRECT_OBJECT", "sulo_predicate": "hasFeature", "target_class": "Language"}
}

vehicle_sulo_mappings = {
    "vehicle_structure_name": {"pattern": "SOLID_LITERAL", "sulo_predicate": "isReferredToIn", "intermediate_class": "InformationObject", "datatype": "http://www.w3.org/2001/XMLSchema#string"},
    "brand_name": {"pattern": "DIRECT_OBJECT", "sulo_predicate": "hasFeature", "target_class": "Brand"},
    "class_name": {"pattern": "DIRECT_OBJECT", "sulo_predicate": "hasFeature", "target_class": "VehicleClass"},
    "serie_name": {"pattern": "DIRECT_OBJECT", "sulo_predicate": "hasFeature", "target_class": "VehicleSeries"},
    "product_group_name": {"pattern": "DIRECT_OBJECT", "sulo_predicate": "hasFeature", "target_class": "ProductGroup"}
}

task_sulo_mappings = {
    "event_date": {"pattern": "SOLID_LITERAL", "sulo_predicate": "atTime", "intermediate_class": "TimeInstant", "datatype": "http://www.w3.org/2001/XMLSchema#dateTime"},
    "task_type": {"pattern": "DIRECT_OBJECT", "sulo_predicate": "hasFeature", "target_class": "TaskType"},
    "task_subject": {"pattern": "SOLID_LITERAL", "sulo_predicate": "isReferredToIn", "intermediate_class": "InformationObject", "datatype": "http://www.w3.org/2001/XMLSchema#string"},
    "task_description": {"pattern": "SOLID_LITERAL", "sulo_predicate": "isReferredToIn", "intermediate_class": "InformationObject", "datatype": "http://www.w3.org/2001/XMLSchema#string"},
    "ishighpriority": {"pattern": "DIRECT_OBJECT", "sulo_predicate": "hasFeature", "target_class": "PriorityLevel"},
    "calldurationinseconds": {"pattern": "SOLID_LITERAL", "sulo_predicate": "atTime", "intermediate_class": "DurationSec", "datatype": "http://www.w3.org/2001/XMLSchema#integer"}
}

sulo_case_to_task_mappings = {
    "task_id": {"pattern": "DIRECT_OBJECT", "sulo_predicate": "hasPart", "target_class": "Task"}
}

survey_sulo_mappings = {
    "numbervalue": {"pattern": "DIRECT_OBJECT", "sulo_predicate": "hasFeature", "target_class": "SurveyRating"}
}

history_sulo_mappings = {
    "event_date": {"pattern": "SOLID_LITERAL", "sulo_predicate": "atTime", "intermediate_class": "TimeInstant", "datatype": "http://www.w3.org/2001/XMLSchema#dateTime"},
    "changed_field": {"pattern": "SOLID_LITERAL", "sulo_predicate": "isReferredToIn", "intermediate_class": "InformationObject", "datatype": "http://www.w3.org/2001/XMLSchema#string"},
    "old_value": {"pattern": "SOLID_LITERAL", "sulo_predicate": "isReferredToIn", "intermediate_class": "InformationObject", "datatype": "http://www.w3.org/2001/XMLSchema#string"},
    "new_value": {"pattern": "SOLID_LITERAL", "sulo_predicate": "isReferredToIn", "intermediate_class": "InformationObject", "datatype": "http://www.w3.org/2001/XMLSchema#string"}
}

sulo_case_to_history_mappings = {
    "modification_id": {"pattern": "DIRECT_OBJECT", "sulo_predicate": "hasPart", "target_class": "CaseModification"}
}

## Populate Sulo

In [ ]:
import pyspark.pandas as ps
import glob
import os
import shutil

volume_base_path = "/Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km"
temp_dir = f"{volume_base_path}/temp_sulo_graph_dir"
merged_dir = f"{volume_base_path}/merged_sulo_graph_dir"
final_file = f"{volume_base_path}/SULO_populated_graph.nt"

shutil.rmtree(temp_dir, ignore_errors=True)
shutil.rmtree(merged_dir, ignore_errors=True)

def process_and_append(df, entity_col, entity_type, mappings, output_dir):
    print(f"Processing {entity_type} domain...")
    triples_list = populate_sulo_kg_explicit(df, entity_col, entity_type, mappings)
    
    if triples_list:
        # Concat ONLY this specific domain to keep the execution plan small
        domain_series = ps.concat(triples_list, ignore_index=True)
        domain_series.to_frame(name="ntriples").to_spark().write.mode("append").text(output_dir)

# By executing these one at a time, the Spark Driver clears its memory after each line.
process_and_append(case_df, "case_id", "Case", case_sulo_mappings, temp_dir)
process_and_append(history_df, "case_id", "Case", sulo_case_to_history_mappings, temp_dir)
process_and_append(survey_df, "case_id", "Case", survey_sulo_mappings, temp_dir)
process_and_append(account_df, "id", "Account", account_sulo_mappings, temp_dir)
process_and_append(task_df, "task_id", "Task", task_sulo_mappings, temp_dir)
process_and_append(task_df, "case_id", "Case", sulo_case_to_task_mappings, temp_dir)
process_and_append(history_df, "modification_id", "CaseModification", history_sulo_mappings, temp_dir)
process_and_append(vehicle_df, "vehicle_structure_id", "Vehicle", vehicle_sulo_mappings, temp_dir)

print("All domains processed. Coalescing into a single file...")

master_spark_df = spark.read.text(temp_dir)

master_spark_df.coalesce(1).write.mode("overwrite").text(merged_dir)

part_file_path = glob.glob(f"{merged_dir}/part-*")[0]

shutil.move(part_file_path, final_file)

shutil.rmtree(temp_dir, ignore_errors=True)
shutil.rmtree(merged_dir, ignore_errors=True)

total_triples = spark.read.text(final_file).count()
print(f"Success! SULO Knowledge Graph generated at: {final_file}")
print(f"Total Triples Generated: {total_triples}")

## Create Balanced Subgraph

In [ ]:
sampled_df = spark.sql("""
    WITH binary_scored_cases AS (
        SELECT DISTINCT 
            c.case_id,
            CASE WHEN CAST(sqr.numbervalue AS FLOAT) >= 3 THEN 1 ELSE 0 END AS satisfaction_score
        FROM ds_ccc.kg_sulo_sample_cases_table c
        JOIN stg_ccc.sf_survey_subject ss ON c.case_id = ss.subjectid
        JOIN stg_ccc.sf_survey_invitation si ON ss.parentid = si.id
        JOIN stg_ccc.sf_survey_question_response sqr ON si.id = sqr.invitationid
        WHERE sqr.questionid = '0Ku7S000000RnrrSAC'
          AND sqr.numbervalue IS NOT NULL
    ),
    sampled_negative AS (
        SELECT case_id FROM binary_scored_cases 
        WHERE satisfaction_score = 0 
        ORDER BY RAND(42) LIMIT 25000
    ),
    sampled_positive AS (
        SELECT case_id FROM binary_scored_cases 
        WHERE satisfaction_score = 1 
        ORDER BY RAND(42) LIMIT 25000
    )
    SELECT case_id FROM sampled_negative
    UNION ALL
    SELECT case_id FROM sampled_positive
""")

target_table = "ds_ccc.kg_sulo_balanced_case_ids"
print(f"Materializing stratified sample to permanent table {target_table}...")

# Materialize once so re-running this cell doesn't draw a different random sample
# sampled_df.write.mode("overwrite").saveAsTable(target_table)

spark.sql(f"SELECT case_id FROM {target_table}").createOrReplaceTempView("cases_with_surveys")

case_filtered = spark.sql("""
    SELECT case_id, person_account_id, business_account_id, case_vehicle_structure_id, case_closure_type, case_type, case_service_type, support_level, customer_country, market_country, case_reason, case_priority, case_description, case_solution_final_statement, case_handling_time_sec, case_closed_date, case_created_date, case_last_modified_date, case_conversation_amt, case_session_amt, is_closed_flg 
    FROM stg_ccc.w_case
    WHERE case_id IN (SELECT case_id FROM cases_with_surveys)
""")

case_filtered.cache()
case_filtered.createOrReplaceTempView("sampled_cases")
subset_case_df = case_filtered.pandas_api()

In [132]:
account_filtered = spark.sql("""
    SELECT a.id, a.salutation, a.type, a.createddate, a.lastmodifieddate, 
           a.core_country__c, a.core_preferredlanguage__c 
    FROM stg_ccc.sf_account a
    WHERE a.isdeleted = 'N'
    AND (
        a.id IN (SELECT person_account_id FROM sampled_cases WHERE person_account_id IS NOT NULL)
        OR 
        a.id IN (SELECT business_account_id FROM sampled_cases WHERE business_account_id IS NOT NULL)
    )
""")

subset_account_df = account_filtered.pandas_api()

vehicle_filtered = spark.sql("SELECT vehicle_structure_id, vehicle_structure_name, brand_name, class_name, serie_name, product_group_name FROM stg_ccc.w_vehicle_structure")

subset_vehicle_df = vehicle_filtered.pandas_api()

survey_case = spark.sql("""
    SELECT 
        c.case_id,
        sqr.id AS survey_response_id,
        sqr.questionid,
        CASE 
            WHEN CAST(sqr.numbervalue AS FLOAT) >= 3 THEN 1 
            ELSE 0 
        END AS numbervalue,
        si.id AS invitation_id
    FROM sampled_cases c
    JOIN stg_ccc.sf_survey_subject ss ON c.case_id = ss.subjectid
    JOIN stg_ccc.sf_survey_invitation si ON ss.parentid = si.id
    JOIN stg_ccc.sf_survey_question_response sqr ON si.id = sqr.invitationid
    WHERE sqr.questionid = '0Ku7S000000RnrrSAC'
""")
subset_survey_df = survey_case.pandas_api()

history_filtered = spark.sql("""
    WITH select_cases AS (
        SELECT case_id
        FROM sampled_cases
    )
    SELECT
        ch.id as modification_id,
        ch.caseid AS case_id,
        ch.createddate AS event_date,
        ch.field AS changed_field,
        ch.oldvalue AS old_value,
        ch.newvalue AS new_value
    FROM 
        westeurope_cac_dwh_prd.sf_ccc.casehistory ch
    WHERE 
        ch.caseid IN (SELECT case_id FROM select_cases)
""")

subset_history_df = history_filtered.pandas_api()

task_filtered = spark.sql("""
    WITH select_cases AS (
        SELECT case_id
        FROM sampled_cases
    )
    SELECT
        t.whatid AS case_id,
        t.completeddatetime AS event_date,
        t.id AS task_id,
        t.type AS task_type,
        t.subject AS task_subject,
        t.description AS task_description,
        t.ishighpriority,
        t.calldurationinseconds
    FROM
        westeurope_cac_dwh_prd.sf_ccc.task t
    WHERE 
        t.whatid IN (SELECT case_id FROM select_cases)
""")

subset_task_df = task_filtered.pandas_api()

In [ ]:
baseline_collection = []

baseline_collection.extend(populate_baseline_kg_explicit(subset_survey_df, "case_id", "Case", survey_mappings))
baseline_collection.extend(populate_baseline_kg_explicit(subset_account_df, "id", "Account", account_mappings))
baseline_collection.extend(populate_baseline_kg_explicit(subset_task_df, "task_id", "Task", task_mappings))
baseline_collection.extend(populate_baseline_kg_explicit(subset_task_df, "case_id", "Case", case_to_task_mappings))
baseline_collection.extend(populate_baseline_kg_explicit(subset_history_df, "modification_id", "CaseModification", history_mappings))
baseline_collection.extend(populate_baseline_kg_explicit(subset_history_df, "case_id", "Case", case_to_history_mappings))
baseline_collection.extend(populate_baseline_kg_explicit(subset_vehicle_df, "vehicle_structure_id", "Vehicle", vehicle_mappings))
baseline_collection.extend(populate_baseline_kg_explicit(subset_case_df, "case_id", "Case", case_mappings))

master_baseline_triples_ps = ps.concat(baseline_collection, ignore_index=True)
master_spark_df = master_baseline_triples_ps.to_frame(name="ntriples").to_spark()

volume_base_path = "/Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km"
temp_dir = f"{volume_base_path}/temp_graph_dir"
final_file = f"{volume_base_path}/subset_balanced_baseline_populated_graph.nt"

print("Coalescing and writing data (this may take a moment)...")
master_spark_df.coalesce(1).write.mode("overwrite").text(temp_dir)
part_file_path = glob.glob(f"{temp_dir}/part-*")[0]

shutil.move(part_file_path, final_file)

shutil.rmtree(temp_dir)

print(f"Success! Single file generated at: {final_file}")

total_triples = master_spark_df.count()
print(f"Baseline Knowledge Graph successfully populated. Total Triples Generated: {total_triples}")

In [ ]:
import pyspark.pandas as ps
import glob
import os
import shutil

volume_base_path = "/Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km"
temp_dir = f"{volume_base_path}/temp_sulo_graph_dir"
merged_dir = f"{volume_base_path}/merged_sulo_graph_dir"
final_file = f"{volume_base_path}/subgraph_balanced_sulo_populated_graph.nt"

shutil.rmtree(temp_dir, ignore_errors=True)
shutil.rmtree(merged_dir, ignore_errors=True)

def process_and_append(df, entity_col, entity_type, mappings, output_dir):
    print(f"Processing {entity_type} domain...")
    triples_list = populate_sulo_kg_explicit(df, entity_col, entity_type, mappings)
    
    if triples_list:
        # Concat ONLY this specific domain to keep the execution plan small
        domain_series = ps.concat(triples_list, ignore_index=True)
        domain_series.to_frame(name="ntriples").to_spark().write.mode("append").text(output_dir)

# By executing these one at a time, the Spark Driver clears its memory after each line.
process_and_append(subset_case_df, "case_id", "Case", case_sulo_mappings, temp_dir)
process_and_append(subset_history_df, "case_id", "Case", sulo_case_to_history_mappings, temp_dir)
process_and_append(subset_survey_df, "case_id", "Case", survey_sulo_mappings, temp_dir)
process_and_append(subset_account_df, "id", "Account", account_sulo_mappings, temp_dir)
process_and_append(subset_task_df, "task_id", "Task", task_sulo_mappings, temp_dir)
process_and_append(subset_task_df, "case_id", "Case", sulo_case_to_task_mappings, temp_dir)
process_and_append(subset_history_df, "modification_id", "CaseModification", history_sulo_mappings, temp_dir)
process_and_append(subset_vehicle_df, "vehicle_structure_id", "Vehicle", vehicle_sulo_mappings, temp_dir)

print("All domains processed. Coalescing into a single file...")

master_spark_df = spark.read.text(temp_dir)

master_spark_df.coalesce(1).write.mode("overwrite").text(merged_dir)

part_file_path = glob.glob(f"{merged_dir}/part-*")[0]

shutil.move(part_file_path, final_file)

shutil.rmtree(temp_dir, ignore_errors=True)
shutil.rmtree(merged_dir, ignore_errors=True)

total_triples = spark.read.text(final_file).count()
print(f"Success! SULO Knowledge Graph generated at: {final_file}")
print(f"Total Triples Generated: {total_triples}")